# FastF1 data exploration — what can the dashboard actually show?

Phase 1. Goal is **not** modeling, it is inventory: for every section we want on the
dashboard (weather / circuit / standings prediction / tyre strategy / safety car /
fun facts), find out which FastF1 object holds the data, what the columns are,
how expensive it is to fetch, and **when in the race weekend it becomes available**.

Everything here uses the 2025 season as the reference season.

| Section | Question |
|---|---|
| 1 | Season schedule — what a race weekend looks like, and when each session runs |
| 2 | Session results — grid, finish, status, points (the prediction labels) |
| 3 | Weather — per-session samples, season-wide summary |
| 4 | Circuit info — corners, rotation, track map |
| 5 | Laps & stints — tyre compounds, stint lengths, degradation |
| 6 | Track status & race control — safety car, VSC, red flags |
| 7 | Telemetry — speed / throttle / DRS, and what it costs to load |
| 8 | Ergast (jolpica) — standings and pit stops that FastF1 timing doesn't expose |
| 9 | Availability map — which section can refresh on Thu / Fri / Sat / Sun |

## 0 · Setup

Cache lives in `data/fastf1_cache/` and is gitignored. First fetch of a session is slow
(network); every later load is local.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "nb":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import fastf1
import fastf1.plotting as f1plt
from fastf1.ergast import Ergast

from src.config import SEASON, enable_cache

enable_cache()

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

In [ ]:
# One reference round used throughout the notebook. Change this to look at another race.
RACE_ROUND = 1

## 1 · Season schedule

`fastf1.get_event_schedule(year)` is the cheapest call in the library — one request, no
session data. It is what the dashboard needs to know *which* weekend is live and *when*
each session starts.

→ feeds the **race week / circuit info** header.

In [ ]:
schedule = fastf1.get_event_schedule(SEASON, include_testing=False)

print(f"{SEASON} season: {len(schedule)} rounds")
print(f"Event formats: {schedule['EventFormat'].value_counts().to_dict()}")
print(f"Columns: {schedule.columns.tolist()}")

schedule[["RoundNumber", "EventName", "Country", "Location", "EventDate", "EventFormat"]]

In [ ]:
# Each event carries 5 named sessions with a local and a UTC datetime.
# The session names differ between conventional and sprint weekends -- this is
# exactly what drives the Thu/Fri/Sat/Sun refresh schedule.
for fmt in schedule["EventFormat"].unique():
    example = schedule.loc[schedule["EventFormat"] == fmt].iloc[0]
    print(f"\n{fmt.upper()} -- e.g. {example['EventName']}")
    for i in range(1, 6):
        name = example[f"Session{i}"]
        start = example[f"Session{i}DateUtc"]
        print(f"  Session{i}: {name:<22} {start} UTC")

In [ ]:
# Split into completed vs upcoming. Only completed rounds have timing data.
today = pd.Timestamp.now(tz="UTC").tz_localize(None)

completed = schedule.loc[schedule["EventDate"] < today].copy()
upcoming = schedule.loc[schedule["EventDate"] >= today].copy()

print(f"{len(completed)} completed, {len(upcoming)} upcoming (as of {today.date()})")
if len(upcoming):
    nxt = upcoming.iloc[0]
    print(f"Next up: R{nxt['RoundNumber']} {nxt['EventName']} on {nxt['EventDate'].date()}")

## 2 · Session results

`Session.results` is a DataFrame that always has the same columns regardless of session
type — irrelevant ones are just empty. This is the label source for the ranking model,
and `GridPosition` is the single strongest feature the demo notebook was missing.

→ feeds **排名预测**.

In [ ]:
session = fastf1.get_session(SEASON, RACE_ROUND, "R")
session.load(telemetry=False)

print(session)
print(f"Total laps: {session.total_laps} | drivers: {len(session.drivers)}")
print(f"Result columns: {session.results.columns.tolist()}")

In [ ]:
result_cols = ["Abbreviation", "FullName", "TeamName", "GridPosition", "Position",
               "ClassifiedPosition", "Status", "Points", "Laps", "Time"]

session.results[result_cols]

In [ ]:
# ClassifiedPosition is the *official* classification and carries the DNF reasons:
# integer = classified, "R" retired, "D" disqualified, "E" excluded, "N" not classified.
# Status says why. Both matter for the SC/incident model and for "did they finish" features.
print(session.results["Status"].value_counts().to_dict())

dnf = session.results.loc[~session.results["ClassifiedPosition"].str.isdigit()]
dnf[["Abbreviation", "TeamName", "ClassifiedPosition", "Status", "Laps"]]

In [ ]:
# Qualifying carries Q1/Q2/Q3 times -- available Saturday, i.e. before the race prediction
# has to be published. Loading results only (no laps/weather/messages) is fast.
quali = fastf1.get_session(SEASON, RACE_ROUND, "Q")
quali.load(laps=False, telemetry=False, weather=False, messages=False)

quali.results[["Abbreviation", "TeamName", "Position", "Q1", "Q2", "Q3"]]

In [ ]:
# Grid vs finish for one race: how much of the result is decided on Saturday?
grid_vs_finish = session.results[["Abbreviation", "GridPosition", "Position"]].dropna()
corr = grid_vs_finish["GridPosition"].corr(grid_vs_finish["Position"], method="spearman")

print(f"Spearman(grid, finish) = {corr:.3f}")
grid_vs_finish.assign(gained=lambda d: d["GridPosition"] - d["Position"]).sort_values("gained", ascending=False)

## 3 · Weather

`Session.weather_data` is one sample per minute for the whole session:
`Time, AirTemp, Humidity, Pressure, Rainfall, TrackTemp, WindDirection, WindSpeed`.

Note this is *observed* weather — FastF1 has no forecast. For the Thursday/Friday
"race week weather" panel we will need an external forecast API and can only use
FastF1 weather as the historical prior (how often is this circuit wet in September?).

→ feeds **比赛周天气**.

In [ ]:
weather = session.weather_data

print(f"{len(weather)} samples, one per minute")
print(f"Columns: {weather.columns.tolist()}")

weather.head()

In [ ]:
weather.describe()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
minutes = weather["Time"].dt.total_seconds() / 60

axes[0].plot(minutes, weather["AirTemp"], label="Air")
axes[0].plot(minutes, weather["TrackTemp"], label="Track")
axes[0].set_ylabel("°C"); axes[0].legend()

axes[1].plot(minutes, weather["Humidity"], color="tab:blue")
axes[1].set_ylabel("Humidity %")

axes[2].plot(minutes, weather["WindSpeed"], color="tab:green")
axes[2].set_ylabel("Wind m/s"); axes[2].set_xlabel("Session minute")

fig.suptitle(f"{session.event['EventName']} {SEASON} — race weather")
plt.tight_layout()
plt.show()

In [ ]:
# Season-wide weather. Weather-only loads skip laps/telemetry/messages so this is
# the cheap way to scan a whole season -- still expect a few minutes on a cold cache.
def race_weather(year, rnd):
    s = fastf1.get_session(year, rnd, "R")
    s.load(laps=False, telemetry=False, weather=True, messages=False)
    w = s.weather_data
    return {
        "round": rnd,
        "event": s.event["EventName"],
        "air_temp": w["AirTemp"].mean(),
        "track_temp_max": w["TrackTemp"].max(),
        "humidity": w["Humidity"].mean(),
        "wind_speed": w["WindSpeed"].mean(),
        "rain_pct": 100 * w["Rainfall"].mean(),
    }

weather_season = pd.DataFrame([race_weather(SEASON, r) for r in completed["RoundNumber"]])
weather_season.round(1)

In [ ]:
# Which 2025 races were actually wet?
wet = weather_season.loc[weather_season["rain_pct"] > 0].sort_values("rain_pct", ascending=False)
print(f"{len(wet)} of {len(weather_season)} races saw rainfall")
wet[["round", "event", "rain_pct", "air_temp", "track_temp_max"]].round(1)

## 4 · Circuit info

`Session.get_circuit_info()` returns corners, marshal lights and marshal sectors as
DataFrames (`X, Y, Number, Letter, Angle, Distance`) plus a `rotation` angle for drawing
the map the right way up. The data comes from MultiViewer — hand-made, good enough for
visuals, not for physics.

`Distance` is only filled once telemetry is loaded.

→ feeds **赛道信息**.

In [ ]:
circuit = session.get_circuit_info()

print(f"{len(circuit.corners)} corners | {len(circuit.marshal_lights)} marshal lights "
      f"| {len(circuit.marshal_sectors)} marshal sectors | rotation {circuit.rotation}°")

circuit.corners.head(10)

In [ ]:
# The track map needs positional telemetry. This is the one expensive load in the
# notebook -- tens of MB per session on a cold cache, instant afterwards.
session_tel = fastf1.get_session(SEASON, RACE_ROUND, "R")
session_tel.load(telemetry=True, weather=False, messages=False)

fastest = session_tel.laps.pick_fastest()
pos = fastest.get_pos_data()

print(f"Fastest lap: {fastest['Driver']} {fastest['LapTime']} on lap {fastest['LapNumber']:.0f}")
print(f"Position samples: {len(pos)} | columns: {pos.columns.tolist()}")

In [ ]:
# Rotate the track into its broadcast orientation and label the corners.
angle = circuit.rotation / 180 * np.pi
rot = np.array([[np.cos(angle), np.sin(angle)], [-np.sin(angle), np.cos(angle)]])

track = np.column_stack([pos["X"], pos["Y"]]) @ rot
corners = np.column_stack([circuit.corners["X"], circuit.corners["Y"]]) @ rot

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(track[:, 0], track[:, 1], color="black", lw=3)
for (x, y), (_, corner) in zip(corners, circuit.corners.iterrows()):
    ax.scatter(x, y, s=120, color="whitesmoke", edgecolor="grey", zorder=3)
    ax.text(x, y, f"{corner['Number']}{corner['Letter']}", ha="center", va="center", fontsize=7, zorder=4)

ax.set_aspect("equal")
ax.axis("off")
ax.set_title(f"{session_tel.event['EventName']} {SEASON} — corner numbering")
plt.show()

## 5 · Laps & tyre strategy

`Session.laps` is the workhorse: one row per driver per lap, with `Compound`, `TyreLife`,
`FreshTyre`, `Stint`, `PitInTime`/`PitOutTime`, sector times, speed traps and
`TrackStatus`. `IsAccurate` flags laps that FastF1 considers clean enough to compare.

→ feeds **最佳轮胎策略**.

In [ ]:
laps = session.laps

print(f"{len(laps)} laps x {len(laps.columns)} columns")
print(f"Columns: {laps.columns.tolist()}")
print(f"Compounds used: {laps['Compound'].value_counts().to_dict()}")
print(f"Accurate laps: {laps['IsAccurate'].sum()} / {len(laps)}")

laps.head()

In [ ]:
# Stint table: one row per driver per stint. This *is* the strategy chart.
stints = (laps.groupby(["Driver", "Stint", "Compound"], as_index=False)
              .agg(start_lap=("LapNumber", "min"),
                   end_lap=("LapNumber", "max"),
                   tyre_laps=("LapNumber", "count"),
                   median_pace=("LapTime", "median")))

stints.sort_values(["Driver", "Stint"]).head(20)

In [ ]:
# Strategy chart -- horizontal stint bars per driver, coloured by compound.
order = session.results["Abbreviation"].dropna().tolist()

fig, ax = plt.subplots(figsize=(10, 8))
for driver in order:
    offset = 0
    for _, stint in stints.loc[stints["Driver"] == driver].sort_values("Stint").iterrows():
        ax.barh(driver, stint["tyre_laps"], left=offset,
                color=f1plt.get_compound_color(stint["Compound"], session=session),
                edgecolor="black", linewidth=0.5)
        offset += stint["tyre_laps"]

ax.invert_yaxis()
ax.set_xlabel("Lap")
ax.set_title(f"{session.event['EventName']} {SEASON} — tyre strategies")
plt.tight_layout()
plt.show()

In [ ]:
# Pace by compound, restricted to quick laps (within 107% of the session best).
by_compound = (laps.pick_quicklaps()
                   .groupby("Compound")["LapTime"]
                   .agg(laps="count", median="median", best="min"))

by_compound

In [ ]:
# Degradation: median quick-lap time against tyre age, per compound.
# This curve is what an "optimal strategy" recommendation has to be built on.
deg = (laps.pick_quicklaps()
           .assign(lap_seconds=lambda d: d["LapTime"].dt.total_seconds())
           .groupby(["Compound", "TyreLife"], as_index=False)["lap_seconds"]
           .median())

fig, ax = plt.subplots(figsize=(9, 5))
for compound, group in deg.groupby("Compound"):
    ax.plot(group["TyreLife"], group["lap_seconds"], marker="o", ms=3, label=compound,
            color=f1plt.get_compound_color(compound, session=session))

ax.set_xlabel("Tyre life (laps)")
ax.set_ylabel("Median lap time (s)")
ax.set_title(f"{session.event['EventName']} {SEASON} — tyre degradation")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Pit stops as seen by the timing data. Ergast (section 8) gives the actual
# stationary duration; FastF1 gives the in/out lap, which is what strategy needs.
box_laps = laps.loc[laps["PitInTime"].notna(), ["Driver", "LapNumber", "Compound", "TyreLife"]]

print(f"{len(box_laps)} pit stops, {box_laps.groupby('Driver').size().mean():.1f} per driver")
box_laps.groupby("LapNumber").size().rename("stops").to_frame().T

## 6 · Track status & race control

Two sources for incidents:

- `Session.track_status` — one row per status change, `Status` is a single digit:
  `1` clear, `2` yellow, `4` safety car, `5` red flag, `6` VSC deployed, `7` VSC ending.
- `Session.race_control_messages` — the actual messages, with
  `Time, Category, Message, Status, Flag, Scope, Sector, RacingNumber, Lap`.

Each lap also carries a `TrackStatus` string concatenating every status active during
that lap (so `"14"` means the lap started clear and went to safety car).

→ feeds **安全车 / 红旗预测**.

In [ ]:
STATUS_LABELS = {"1": "AllClear", "2": "Yellow", "4": "SafetyCar",
                 "5": "RedFlag", "6": "VSCDeployed", "7": "VSCEnding"}

track_status = session.track_status.copy()
track_status["Label"] = track_status["Status"].map(STATUS_LABELS)

print(f"{len(track_status)} status changes")
track_status

In [ ]:
# Which laps ran under which status.
for code, label in [("4", "Safety car"), ("6", "VSC"), ("5", "Red flag"), ("2", "Yellow")]:
    affected = sorted(laps.loc[laps["TrackStatus"].str.contains(code, na=False), "LapNumber"].unique())
    print(f"{label:<12} {len(affected):>3} laps  {affected}")

In [ ]:
rcm = session.race_control_messages

print(f"{len(rcm)} messages | categories: {rcm['Category'].value_counts().to_dict()}")
rcm.loc[rcm["Category"] == "Flag", ["Time", "Lap", "Flag", "Scope", "Sector", "Message"]].head(20)

In [ ]:
# Incident messages are free-text -- useful for the fun-fact section and as weak labels.
incident_words = "INCIDENT|PENALTY|INVESTIGATION|DELETED|CAR \\d+"
rcm.loc[rcm["Message"].str.contains(incident_words, case=False, na=False),
        ["Lap", "Category", "Message"]].head(20)

In [ ]:
# Season-wide incident counts -- the label set for a safety car / red flag model.
# Needs laps + messages, so this is the slowest scan in the notebook.
def race_incidents(year, rnd):
    s = fastf1.get_session(year, rnd, "R")
    s.load(laps=True, telemetry=False, weather=False, messages=True)
    ts = s.track_status
    return {
        "round": rnd,
        "event": s.event["EventName"],
        "total_laps": s.total_laps,
        "sc_periods": int((ts["Status"] == "4").sum()),
        "vsc_periods": int((ts["Status"] == "6").sum()),
        "red_flags": int((ts["Status"] == "5").sum()),
        "yellows": int((ts["Status"] == "2").sum()),
        "dnfs": int((~s.results["ClassifiedPosition"].str.isdigit()).sum()),
    }

incidents = pd.DataFrame([race_incidents(SEASON, r) for r in completed["RoundNumber"]])
incidents

In [ ]:
# Base rates -- the number any SC prediction has to beat.
print(f"Races with >=1 safety car: {(incidents['sc_periods'] > 0).mean():.0%}")
print(f"Races with >=1 VSC:        {(incidents['vsc_periods'] > 0).mean():.0%}")
print(f"Races with a red flag:     {(incidents['red_flags'] > 0).mean():.0%}")
print(f"Mean DNFs per race:        {incidents['dnfs'].mean():.1f}")

incidents.sort_values("sc_periods", ascending=False).head(10)[
    ["round", "event", "sc_periods", "vsc_periods", "red_flags", "dnfs"]]

## 7 · Telemetry

`Session.car_data` and `Session.pos_data` are dicts keyed by driver number, ~4 Hz for the
whole session. Rich, but heavy — this is the reason `telemetry=False` is the default in
every other cell here. Reuses `session_tel` loaded in section 4.

→ feeds **fun facts** (top speed, full-throttle share, DRS usage).

In [ ]:
car = session_tel.car_data
pos = session_tel.pos_data

sample_driver = session_tel.results.index[0]
print(f"Drivers with telemetry: {len(car)}")
print(f"car_data columns: {car[sample_driver].columns.tolist()}")
print(f"pos_data columns: {pos[sample_driver].columns.tolist()}")
print(f"Samples for driver {sample_driver}: {len(car[sample_driver]):,}")

In [ ]:
# Fun-fact material: top speed and full-throttle share on each driver's fastest lap.
rows = []
for number in session_tel.drivers:
    driver_laps = session_tel.laps.pick_drivers(number)
    fastest_lap = driver_laps.pick_fastest()
    if fastest_lap is None or pd.isna(fastest_lap["LapTime"]):
        continue
    tel = fastest_lap.get_car_data()
    rows.append({
        "driver": session_tel.get_driver(number)["Abbreviation"],
        "top_speed": tel["Speed"].max(),
        "full_throttle_pct": 100 * (tel["Throttle"] >= 99).mean(),
        "brake_pct": 100 * tel["Brake"].mean(),
        "drs_pct": 100 * tel["DRS"].isin([10, 12, 14]).mean(),
    })

pd.DataFrame(rows).sort_values("top_speed", ascending=False).round(1)

## 8 · Ergast / jolpica

`fastf1.ergast.Ergast` now points at the jolpica mirror (`api.jolpi.ca`) since ergast.com
was retired. It gives two things FastF1's own timing data does not expose neatly:
**championship standings** and **pit stop durations**.

It is rate limited, so batch and cache anything the dashboard needs.

→ feeds **排名预测** (standings context) and **fun facts** (fastest stop).

In [ ]:
ergast = Ergast(result_type="pandas", auto_cast=True)

standings = ergast.get_driver_standings(season=SEASON).content[0]
print(f"Columns: {standings.columns.tolist()}")

standings[["position", "driverCode", "givenName", "familyName", "constructorNames", "points", "wins"]]

In [ ]:
constructors = ergast.get_constructor_standings(season=SEASON).content[0]
constructors[["position", "constructorName", "points", "wins"]]

In [ ]:
# Pit stop durations -- the actual stationary time, not available from FastF1 timing.
# Default page limit is 30 -- a race has ~40 stops, so raise it.
pit_stops = ergast.get_pit_stops(season=SEASON, round=RACE_ROUND, limit=200).content[0]

print(f"{len(pit_stops)} stops | columns: {pit_stops.columns.tolist()}")
pit_stops.sort_values("duration").head(10)

In [ ]:
# Race results via Ergast for comparison: same race, driverId/constructorId keys that
# join against historical seasons. The demo notebook built its whole feature set on this.
ergast_results = ergast.get_race_results(season=SEASON, round=RACE_ROUND).content[0]
print(f"Columns: {ergast_results.columns.tolist()}")

ergast_results[["position", "driverId", "constructorId", "grid", "laps", "points", "status"]]

## 9 · Availability map

The dashboard refreshes Thursday → Sunday. Not every section has data on every day, and
getting this wrong means publishing a stale or empty panel. This table is the contract
the Phase 2 pipeline has to honour.

In [ ]:
availability = pd.DataFrame([
    # section,              source,                          thu,   fri,   sat,   sun
    ("Race week header",    "get_event_schedule",            True,  True,  True,  True),
    ("Circuit info",        "Session.get_circuit_info",      True,  True,  True,  True),
    ("Weather (forecast)",  "external forecast API",         True,  True,  True,  True),
    ("Weather (observed)",  "Session.weather_data",          False, True,  True,  True),
    ("Practice pace",       "Session.laps (FP1-FP3)",        False, True,  True,  True),
    ("Grid",                "Session('Q').results",          False, False, True,  True),
    ("Ranking prediction",  "model on schedule+quali",       True,  True,  True,  True),
    ("Tyre strategy",       "Session.laps Compound/Stint",   False, True,  True,  True),
    ("Safety car forecast", "history + circuit priors",      True,  True,  True,  True),
    ("Fun facts",           "telemetry + Ergast + history",  True,  True,  True,  True),
], columns=["section", "source", "thu", "fri", "sat", "sun"])

availability

In [ ]:
# Cost of each source, so the Phase 2 pipeline knows what it can afford per refresh.
cost = pd.DataFrame([
    ("get_event_schedule",       "1 request",        "seconds"),
    ("Session.results only",     "load(laps=False, telemetry=False, weather=False, messages=False)", "seconds"),
    ("Session weather only",     "load(laps=False, telemetry=False, weather=True, messages=False)",  "~10s cold"),
    ("Session laps + messages",  "load(telemetry=False)",  "~30s cold"),
    ("Session full telemetry",   "load(telemetry=True)",   "minutes, tens of MB"),
    ("Ergast standings",         "1 request, rate limited", "seconds"),
], columns=["source", "call", "cost"])

cost

### Open questions for Phase 2

- **Weather forecast** has no FastF1 source. Need an external API (Open-Meteo is free and
  needs no key) keyed on the circuit lat/lon — which Ergast `get_circuits` provides.
- **Tyre allocation** (which compounds Pirelli brings to each round) is not in FastF1 at
  all. Either scrape it or hardcode per round.
- **Overtakes** have no direct field. Closest proxy is lap-by-lap position change from
  `Session.laps["Position"]`, which counts pit cycles as overtakes — needs filtering.
- **Sprint weekends** shift the session layout, so anything keyed on "Session4 == Qualifying"
  will break. Use `Event.get_qualifying()` / `get_race()` instead of session numbers.
- **Cold cache on a fresh server**: a full 2025 scan with telemetry is large. Decide what the
  deployed job caches vs. recomputes.